In [1]:
# =========== ЗАПОЛНИТЕ ЭТИ ТРИ ПОЛЯ ===========
STUDENT_NAME  = "Тимошенко Софья Антоновна"
STUDENT_GROUP = "ИНФА-231"
VARIANT_NO    = 12
# ==============================================

VARIANTS = {
    n: {"random_state": 100 + n,
        "N": [500, 750, 1000, 1250, 1500][(n - 1) % 5],
        "min_len": [2, 3, 4, 3, 2][(n - 1) % 5],
        "operator": ["AND", "OR", "AND", "AND", "OR"][(n - 1) % 5]}
    for n in range(1, 26)
}

assert 1 <= VARIANT_NO <= 25, "Номер варианта — целое число от 1 до 25"
V = VARIANTS[VARIANT_NO]

RANDOM_STATE = V["random_state"]
N            = V["N"]
MIN_LEN      = V["min_len"]
OPERATOR     = V["operator"]
QUERY        = "summer wedding"   # при желании замените на свой запрос из двух-трёх слов

print(f"Студент      : {STUDENT_NAME}, группа {STUDENT_GROUP}")
print(f"Вариант      : {VARIANT_NO}")
print(f"random_state : {RANDOM_STATE}")
print(f"N            : {N}")
print(f"min_len      : {MIN_LEN}")
print(f"Оператор     : {OPERATOR}")
print(f"Запрос       : {QUERY!r}")

Студент      : Тимошенко Софья Антоновна, группа ИНФА-231
Вариант      : 12
random_state : 112
N            : 750
min_len      : 3
Оператор     : OR
Запрос       : 'summer wedding'


In [2]:
import json
import random
import re
import time
import statistics
from collections import defaultdict, Counter
from pathlib import Path

WORK = Path("/content") if Path("/content").exists() else Path(".")
print("Рабочий каталог:", WORK)

Рабочий каталог: /content


Чтение JSONL

In [5]:
def load_documents(path) -> list[dict]:
    """Читает JSONL-корпус и возвращает список документов."""
    documents = []

    with open(path, "r", encoding="utf-8") as f:
        for source_id, line in enumerate(f):
            line = line.strip()
            if not line:
                continue

            obj = json.loads(line)
            result = obj.get("result")

            if obj.get("error") is not None or not isinstance(result, dict):
                continue

            title = result.get("title", "")
            description = result.get("description", "")
            genre = result.get("genre", "")

            text = " ".join(
                str(value).strip()
                for value in (title, description, genre)
                if value
            )

            documents.append({
                "id": source_id,
                "text": text
            })

    return documents


# Самопроверка на маленьком файле
demo = WORK / "demo.jsonl"
demo.write_text(
    '{"tries":0,"result":{"title":"Magic Book","description":"A world of magic","genre":"Fantasy","price":10.0},"error":null}\n'
    '\n'
    '{"tries":1,"result":null,"error":"timeout"}\n',
    encoding="utf-8",
)

_d = load_documents(demo)
print(_d)

assert len(_d) == 1, "Строка с ошибкой и пустая строка должны быть пропущены"
assert _d[0]["text"] == "Magic Book A world of magic Fantasy"
print("Проверка пройдена!")

[{'id': 0, 'text': 'Magic Book A world of magic Fantasy'}]
Проверка пройдена!


Сбор корпуса

In [8]:
GENRES = ["Fantasy", "Science Fiction", "Mystery", "Historical Fiction", "Poetry",
          "Travel", "Philosophy", "Biography", "Childrens", "Romance",
          "Thriller", "Horror", "Classics", "Art", "Music", "Science", "History"]

# Ядро словаря: частотные служебные и содержательные слова, в том числе короткие —
# именно они отсекаются фильтром min_len.
CORE = """the of and to in is it on at by or as be he she we my an for with from
that this not are was were has have had who what when where why how all any
book story world magic life time day night man woman child house door road
ancient kingdom shadow light forest river mountain ocean secret journey voyage
island castle dragon knight witch wizard spell memory silence winter summer
autumn garden letter mirror clock window science machine planet station engineer
signal orbit gravity comet robot murder detective evidence witness alibi verdict
prison escape chase clue love promise wedding stranger neighbour village harbour
lighthouse cottage history empire revolution treaty battle general soldier
archive manuscript museum poem verse rhythm metaphor language grammar dictionary
translation accent teacher student lesson school library reading writing question
answer knowledge map compass railway desert canyon glacier volcano jungle savanna""".split()


def _make_vocabulary(n_tail: int = 6000, seed: int = 7) -> list[str]:
    """CORE + процедурно порождённый «хвост» редких слов разной длины."""
    rng = random.Random(seed)
    cons = "bcdfghjklmnprstvwz"
    vows = "aeiouy"
    tail, seen = [], set(CORE)
    while len(tail) < n_tail:
        syllables = rng.randint(1, 4)
        w = "".join(rng.choice(cons) + rng.choice(vows) +
                    (rng.choice(cons) if rng.random() < 0.45 else "")
                    for _ in range(syllables))
        if 2 <= len(w) <= 12 and w not in seen:
            seen.add(w)
            tail.append(w)
    return CORE + tail


VOCAB = _make_vocabulary()
# Веса по закону Ципфа: слово ранга r встречается примерно в 1/r раз реже первого.
ZIPF_WEIGHTS = [1.0 / (r + 2.5) ** 1.07 for r in range(1, len(VOCAB) + 1)]


def make_synthetic_corpus(n_docs: int, seed: int = 2026) -> list[dict]:
    """Детерминированный учебный корпус с ципфовским распределением слов.

    Работает без сети, даёт любое число документов и воспроизводит два свойства
    настоящего текста: сублинейный рост словаря (закон Хипса) и большую долю слов,
    встретившихся ровно один раз.
    """
    rng = random.Random(seed)
    head = VOCAB[:400]                       # частотное ядро для заголовков
    docs = []
    for i in range(n_docs):
        title = " ".join(w.capitalize() for w in rng.sample(head, rng.randint(2, 4)))
        body = rng.choices(VOCAB, weights=ZIPF_WEIGHTS, k=rng.randint(60, 140))
        description = " ".join(body).capitalize() + "."
        genre = rng.choice(GENRES)
        docs.append({
            "source_id": i,
            "title": title,
            "description": description,
            "genre": genre,
            "price": round(rng.uniform(10, 60), 2),
            "text": f"{title} {description} {genre}",
        })
    return docs


def scrape_books_toscrape(limit: int = 1000, timeout: int = 20) -> list[dict]:
    """Сбор корпуса с учебного сайта books.toscrape.com (только стандартная библиотека)."""
    from urllib.request import urlopen
    from urllib.parse import urljoin
    from html.parser import HTMLParser

    BASE = "https://books.toscrape.com/catalogue/"
    docs, page = [], 1

    def fetch(url):
        with urlopen(url, timeout=timeout) as r:
            return r.read().decode("utf-8", errors="ignore")

    while len(docs) < limit and page <= 50:
        html = fetch(f"{BASE}page-{page}.html")
        hrefs = re.findall(r'<h3><a href="([^"]+)"', html)
        for href in hrefs:
            if len(docs) >= limit:
                break
            book = fetch(urljoin(BASE, href))
            title = re.search(r"<h1>(.*?)</h1>", book)
            desc = re.search(r'<div id="product_description".*?<p>(.*?)</p>', book, re.S)
            genre = re.findall(r'<li><a href="\.\./category/books/[^"]+">(.*?)</a></li>', book)
            price = re.search(r"£([\d.]+)", book)
            docs.append({
                "source_id": len(docs),
                "title": (title.group(1) if title else "").strip(),
                "description": re.sub(r"<[^>]+>", " ", desc.group(1)) if desc else "",
                "genre": genre[0].strip() if genre else "",
                "price": float(price.group(1)) if price else None,
                "text": "",
            })
            docs[-1]["text"] = (f'{docs[-1]["title"]} {docs[-1]["description"]} '
                                f'{docs[-1]["genre"]}')
        page += 1
    return docs


def get_corpus(min_docs: int) -> tuple[list[dict], str]:
    """Возвращает (корпус, название источника), гарантируя размер не меньше min_docs."""
    jsonl = WORK / "result.jsonl"
    if jsonl.exists():
        docs = load_documents(jsonl)
        source = "result.jsonl (семинар 02)"
        print(f"Источник: {source}, документов {len(docs)}")
    else:
        try:
            print("Файл result.jsonl не найден, пробую собрать books.toscrape.com ...")
            docs = scrape_books_toscrape(limit=1000)
            source = "books.toscrape.com"
            print(f"Источник: {source}, документов {len(docs)}")
        except Exception as exc:
            print(f"Сбор не удался ({type(exc).__name__}), перехожу к синтетическому корпусу.")
            docs, source = [], "синтетический корпус"

    if len(docs) < min_docs:
        need = min_docs - len(docs) + 200          # запас для экспериментов
        if docs:
            print(f"ВНИМАНИЕ: документов {len(docs)}, требуется {min_docs}. "
                  f"Дополняю {need} синтетическими.")
            source += " + синтетический корпус"
        extra = make_synthetic_corpus(need)
        for d in extra:
            d["source_id"] += len(docs)
        docs = docs + extra

    return docs, source


t0 = time.perf_counter()
docs_all, CORPUS_SOURCE = get_corpus(min_docs=N + 200)
corpus_time = time.perf_counter() - t0

print(f"\nВсего документов : {len(docs_all)}")
print(f"Источник         : {CORPUS_SOURCE}")
print(f"Время подготовки : {corpus_time:.2f} с")
print("\nПример документа:")
print(json.dumps(docs_all[0], ensure_ascii=False, indent=2)[:400])

Файл result.jsonl не найден, пробую собрать books.toscrape.com ...
Источник: books.toscrape.com, документов 1000

Всего документов : 1000
Источник         : books.toscrape.com
Время подготовки : 148.28 с

Пример документа:
{
  "source_id": 0,
  "title": "A Light in the Attic",
  "description": "It's hard to imagine a world without A Light in the Attic. This now-classic collection of poetry and drawings from Shel Silverstein celebrates its 20th anniversary with this special edition. Silverstein's humorous and creative verse can amuse the dowdiest of readers. Lemon-faced adults and fidgety kids sit still and read thes


Токенизация

In [9]:
TOKEN_RE = re.compile(r"[A-Za-z]+(?:'[A-Za-z]+)?")


def tokenize(text: str, min_len: int = 2) -> list[str]:
    """Нижний регистр -> выделение слов -> фильтрация по длине."""
    if not isinstance(text, str):
        return []

    text = text.lower()
    tokens = TOKEN_RE.findall(text)

    return [word for word in tokens if len(word) >= min_len]


print(tokenize("Magic World, the Ancient Kingdom!", min_len=2))
print(tokenize("Magic World, the Ancient Kingdom!", min_len=4))

assert tokenize("A Magic World", 2) == ["magic", "world"]
assert tokenize("A Magic World", 6) == []
assert tokenize(None, 2) == []
assert tokenize("Don't stop", 2) == ["don't", "stop"]

['magic', 'world', 'the', 'ancient', 'kingdom']
['magic', 'world', 'ancient', 'kingdom']


Воспроизводимая выборка

In [10]:
def sample_documents(documents: list[dict], n: int, random_state: int) -> list[dict]:
    """Воспроизводимая случайная выборка из корпуса."""
    rng = random.Random(random_state)
    if n > len(documents):
        raise ValueError("Недостаточно документов для выборки")
    return rng.sample(documents, n)


docs = sample_documents(docs_all, n=N, random_state=RANDOM_STATE)
print(f"В выборке {len(docs)} документов")

# воспроизводимость: тот же random_state -> та же выборка
again = sample_documents(docs_all, n=N, random_state=RANDOM_STATE)
assert [d["source_id"] for d in docs] == [d["source_id"] for d in again]
other = sample_documents(docs_all, n=N, random_state=RANDOM_STATE + 1)
assert [d["source_id"] for d in docs] != [d["source_id"] for d in other]
assert len(docs) == N

В выборке 750 документов


Инвертированный индекс


In [11]:
def build_inverted_index(documents: list[dict], min_len: int) -> dict[str, set[int]]:
    """Строит индекс: термин -> множество идентификаторов документов."""
    index: dict[str, set[int]] = defaultdict(set)
    for doc_id, doc in enumerate(documents):
        tokens = tokenize(doc.get("text", ""), min_len)
        for term in set(tokens):
            index[term].add(doc_id)
    return dict(index)


t0 = time.perf_counter()
index = build_inverted_index(docs, MIN_LEN)
BUILD_TIME = time.perf_counter() - t0

print(f"Размер словаря           : {len(index)}")
print(f"Время построения индекса : {BUILD_TIME:.6f} с")
print(f"Всего вхождений          : {sum(len(v) for v in index.values())}")

sample_term = sorted(index)[len(index) // 2]
print(f"\nПример: {sample_term!r} -> {sorted(index[sample_term])[:10]} ...")

assert isinstance(index, dict) and index
assert all(isinstance(v, set) for v in index.values())
assert all(len(t) >= MIN_LEN for t in index), "В индексе не должно быть коротких токенов"
assert max(max(v) for v in index.values()) < len(docs)

Размер словаря           : 15967
Время построения индекса : 0.145997 с
Всего вхождений          : 81043

Пример: 'knocking' -> [168, 626] ...


Что внутри словаря

In [12]:
df_pairs = sorted(((len(ids), term) for term, ids in index.items()), reverse=True)

print("Самые частые термины (документная частота):")
for freq, term in df_pairs[:10]:
    print(f"  {term:<18} df = {freq:>5}  ({freq / len(docs):.1%} документов)")

hapax = sum(1 for freq, _ in df_pairs if freq == 1)
print(f"\nТерминов, встретившихся ровно в одном документе: {hapax} "
      f"({hapax / len(index):.1%} словаря)")

Самые частые термины (документная частота):
  the                df =   743  (99.1% документов)
  and                df =   736  (98.1% документов)
  more               df =   705  (94.0% документов)
  that               df =   542  (72.3% документов)
  for                df =   537  (71.6% документов)
  with               df =   534  (71.2% документов)
  from               df =   441  (58.8% документов)
  but                df =   410  (54.7% документов)
  has                df =   363  (48.4% документов)
  this               df =   356  (47.5% документов)

Терминов, встретившихся ровно в одном документе: 8685 (54.4% словаря)


OR

In [14]:
def boolean_search(query: str, index: dict[str, set[int]],
                   operator: str, min_len: int) -> tuple[list[str], list[int]]:
    """Булев поиск по инвертированному индексу."""

    terms = tokenize(query, min_len)
    if not terms:
        return [], []

    if operator not in ("AND", "OR"):
        raise ValueError("Неизвестный оператор")

    result = set(index.get(terms[0], set()))

    for term in terms[1:]:
        posting = index.get(term, set())
        if operator == "AND":
            result.intersection_update(posting)
        else:
            result.update(posting)

    return terms, sorted(result)


# --- проверка на игрушечном индексе из лекции
toy = {"magic": {2, 8, 15}, "world": {2, 5, 8, 21}}
assert boolean_search("magic world", toy, "AND", 2) == (["magic", "world"], [2, 8])
assert boolean_search("magic world", toy, "OR", 2)  == (["magic", "world"], [2, 5, 8, 15, 21])
assert boolean_search("magic dragon", toy, "AND", 2)[1] == []      # dragon нет в индексе
assert boolean_search("magic dragon", toy, "OR", 2)[1] == [2, 8, 15]
assert boolean_search("", toy, "AND", 2) == ([], [])

try:
    boolean_search("magic", toy, "XOR", 2)
    raise AssertionError("Должно быть выброшено ValueError")
except ValueError:
    pass

print("Все проверки пройдены")

Все проверки пройдены


Индекс против полного перебора

In [15]:
def linear_search(query: str, documents: list[dict],
                  operator: str, min_len: int) -> list[int]:
    """Тот же поиск, но полным просмотром корпуса — без индекса."""
    terms = tokenize(query, min_len)
    if not terms:
        return []

    if operator not in ("AND", "OR"):
        raise ValueError("Неизвестный оператор")

    result = []

    for doc_id, doc in enumerate(documents):
        doc_terms = set(tokenize(doc.get("text", ""), min_len))

        if operator == "AND":
            match = all(term in doc_terms for term in terms)
        else:
            match = any(term in doc_terms for term in terms)

        if match:
            result.append(doc_id)

    return result


def median_time(fn, repeats: int = 20) -> float:
    ts = []
    for _ in range(repeats):
        t0 = time.perf_counter()
        fn()
        ts.append(time.perf_counter() - t0)
    return statistics.median(ts)


t_index  = median_time(lambda: boolean_search(QUERY, index, OPERATOR, MIN_LEN))
t_linear = median_time(lambda: linear_search(QUERY, docs, OPERATOR, MIN_LEN))

ids_index  = boolean_search(QUERY, index, OPERATOR, MIN_LEN)[1]
ids_linear = linear_search(QUERY, docs, OPERATOR, MIN_LEN)

print(f"Поиск по индексу : {t_index * 1e6:9.1f} мкс")
print(f"Полный перебор   : {t_linear * 1e6:9.1f} мкс")
print(f"Ускорение        : {t_linear / t_index:9.0f}x")
print(f"Результаты совпадают: {ids_index == ids_linear}")

assert ids_index == ids_linear, "Индекс и полный перебор обязаны давать одинаковый ответ"
assert t_linear > t_index, "Поиск по индексу должен быть быстрее"

Поиск по индексу :       8.2 мкс
Полный перебор   :  160329.6 мкс
Ускорение        :     19581x
Результаты совпадают: True


**Вопрос 2.3.** Ответьте письменно: почему поиск по индексу быстрее и за счёт чего именно?
Какую цену мы заплатили за это ускорение?

In [18]:

ANSWER_2_3 = """
Поиск по инвертированному индексу быстрее, потому что не требуется
каждый раз просматривать тексты всех документов.
Индекс заранее хранит соответствие между каждым термином и списком
документов, в которых он встречается.
Благодаря этому поиск сводится к быстрому обращению к словарю и
выполнению операций над множествами идентификаторов документов.
Ценой ускорения стали дополнительные затраты памяти на хранение индекса,
а также время и вычислительные ресурсы, необходимые для его построения
и обновления. Кроме того, при изменении документов индекс необходимо
поддерживать в актуальном состоянии.
"""
print(ANSWER_2_3)


Поиск по инвертированному индексу быстрее, потому что не требуется 
каждый раз просматривать тексты всех документов. 
Индекс заранее хранит соответствие между каждым термином и списком 
документов, в которых он встречается. 
Благодаря этому поиск сводится к быстрому обращению к словарю и 
выполнению операций над множествами идентификаторов документов. 
Ценой ускорения стали дополнительные затраты памяти на хранение индекса, 
а также время и вычислительные ресурсы, необходимые для его построения 
и обновления. Кроме того, при изменении документов индекс необходимо 
поддерживать в актуальном состоянии.



Прогон варианта

In [19]:
terms_and, ids_and = boolean_search(QUERY, index, "AND", MIN_LEN)
t_and = median_time(lambda: boolean_search(QUERY, index, "AND", MIN_LEN))

terms_or, ids_or = boolean_search(QUERY, index, "OR", MIN_LEN)
t_or = median_time(lambda: boolean_search(QUERY, index, "OR", MIN_LEN))

print(f"Запрос         : {QUERY!r}")
print(f"Термины запроса: {terms_and}")
print()
print(f"AND: найдено {len(ids_and):>5} документов, время {t_and * 1e6:.1f} мкс")
print(f"OR : найдено {len(ids_or):>5} документов, время {t_or * 1e6:.1f} мкс")
print()
print("Первые 10 документов основной выдачи "
      f"({OPERATOR}):")
main_ids = ids_and if OPERATOR == "AND" else ids_or
for doc_id in main_ids[:10]:
    print(f"  [{doc_id:>5}] {docs[doc_id]['title'][:60]}  ({docs[doc_id]['genre']})")

assert set(ids_and) <= set(ids_or), "AND-выдача обязана быть подмножеством OR-выдачи"

Запрос         : 'summer wedding'
Термины запроса: ['summer', 'wedding']

AND: найдено     2 документов, время 3.1 мкс
OR : найдено    38 документов, время 4.3 мкс

Первые 10 документов основной выдачи (OR):
  [    9] The Alien Club  ()
  [   16] My Life Next Door (My Life Next Door )  ()
  [   33] Secrets and Lace (Fatal Hearts #1)  ()
  [   57] Rip it Up and Start Again  ()
  [  104] Lumberjanes, Vol. 1: Beware the Kitten Holy (Lumberjanes #1-  ()
  [  108] Girl Online On Tour (Girl Online #2)  ()
  [  140] The Vacationers  ()
  [  159] This One Summer  ()
  [  180] Harry Potter and the Half-Blood Prince (Harry Potter #6)  ()
  [  237] The Paris Wife  ()


Упражнение 3.2. Влияние min_len и размера корпуса

In [20]:

by_min_len = {}
for ml in [1, 2, 3, 4, 5]:
    t0 = time.perf_counter()
    idx = build_inverted_index(docs, ml)
    elapsed = time.perf_counter() - t0
    by_min_len[ml] = (len(idx), elapsed)

print(f"{'min_len':>8}{'словарь':>12}{'время, с':>12}")
for ml, (vocab, t) in sorted(by_min_len.items()):
    print(f"{ml:>8}{vocab:>12}{t:>12.4f}")

by_size = {}
for size in sorted({100, 250, 500, N}):
    sample = sample_documents(
        docs_all, n=size, random_state=RANDOM_STATE
    )
    t0 = time.perf_counter()
    idx = build_inverted_index(sample, MIN_LEN)
    elapsed = time.perf_counter() - t0
    by_size[size] = (len(idx), elapsed)

print(f"\n{'N':>8}{'словарь':>12}{'время, с':>12}{'слов на документ':>20}")
for size, (vocab, t) in sorted(by_size.items()):
    print(f"{size:>8}{vocab:>12}{t:>12.4f}{vocab / size:>20.2f}")

assert set(by_min_len) == {1, 2, 3, 4, 5}
assert by_min_len[1][0] >= by_min_len[5][0], \
    "С ростом min_len словарь не должен увеличиваться"
assert len(by_size) >= 3

 min_len     словарь    время, с
       1       16130      0.3450
       2       16104      0.2283
       3       15967      0.2337
       4       15447      0.2257
       5       14107      0.4414

       N     словарь    время, с    слов на документ
     100        4894      0.0340               48.94
     250        8657      0.0767               34.63
     500       12599      0.1176               25.20
     750       15967      0.1362               21.29


Сохраниение результатов

In [21]:
INDEX_FILE  = WORK / "index.json"
RESULT_FILE = WORK / "lab_01_results.json"

INDEX_FILE.write_text(
    json.dumps({t: sorted(ids) for t, ids in sorted(index.items())},
               ensure_ascii=False, indent=2),
    encoding="utf-8",
)

report = {
    "student": STUDENT_NAME,
    "group": STUDENT_GROUP,
    "variant": VARIANT_NO,
    "corpus_source": CORPUS_SOURCE,
    "corpus_total": len(docs_all),
    "sample_size": len(docs),
    "random_state": RANDOM_STATE,
    "min_len": MIN_LEN,
    "vocabulary_size": len(index),
    "build_time_seconds": BUILD_TIME,
    "query": QUERY,
    "query_terms": terms_and,
    "main_operator": OPERATOR,
    "and_results_count": len(ids_and),
    "and_time_seconds": t_and,
    "or_results_count": len(ids_or),
    "or_time_seconds": t_or,
    "linear_search_time_seconds": t_linear,
    "index_speedup": t_linear / t_index,
    "vocabulary_by_min_len": {str(k): v[0] for k, v in sorted(by_min_len.items())},
    "vocabulary_by_corpus_size": {str(k): v[0] for k, v in sorted(by_size.items())},
    "results": [
        {"doc_id": i, "source_id": docs[i]["source_id"],
         "title": docs[i]["title"], "genre": docs[i]["genre"], "price": docs[i]["price"]}
        for i in main_ids
    ],
}
RESULT_FILE.write_text(json.dumps(report, ensure_ascii=False, indent=2), encoding="utf-8")

print(f"Сохранено: {INDEX_FILE.name} ({INDEX_FILE.stat().st_size / 1024:.1f} КиБ)")
print(f"Сохранено: {RESULT_FILE.name}")
print()
print("=" * 62)
print("ТАБЛИЦА РЕЗУЛЬТАТОВ (перенесите её в README.md репозитория)")
print("=" * 62)
rows = [
    ("Студент", STUDENT_NAME), ("Группа", STUDENT_GROUP), ("Вариант", VARIANT_NO),
    ("Источник корпуса", CORPUS_SOURCE),
    ("Число документов N", len(docs)), ("random_state", RANDOM_STATE),
    ("min_len", MIN_LEN), ("Размер словаря", len(index)),
    ("Время построения индекса, с", f"{BUILD_TIME:.6f}"),
    ("Запрос", QUERY),
    ("Число результатов AND", len(ids_and)), ("Время AND, с", f"{t_and:.8f}"),
    ("Число результатов OR", len(ids_or)),  ("Время OR, с",  f"{t_or:.8f}"),
    ("Ускорение против перебора", f"{t_linear / t_index:.0f}x"),
]
for k, v in rows:
    print(f"| {k:<30} | {str(v):>22} |")

Сохранено: index.json (1001.0 КиБ)
Сохранено: lab_01_results.json

ТАБЛИЦА РЕЗУЛЬТАТОВ (перенесите её в README.md репозитория)
| Студент                        | Тимошенко Софья Антоновна |
| Группа                         |               ИНФА-231 |
| Вариант                        |                     12 |
| Источник корпуса               |     books.toscrape.com |
| Число документов N             |                    750 |
| random_state                   |                    112 |
| min_len                        |                      3 |
| Размер словаря                 |                  15967 |
| Время построения индекса, с    |               0.145997 |
| Запрос                         |         summer wedding |
| Число результатов AND          |                      2 |
| Время AND, с                   |             0.00000305 |
| Число результатов OR           |                     38 |
| Время OR, с                    |             0.00000431 |
| Ускорение против перебора   

Вывод

In [23]:
CONCLUSION = """
В ходе лабораторной работы был построен инвертированный индекс для корпуса
из 750 документов. При значении min_len = 3 размер словаря составил 15967
терминов, а время построения индекса — около 0.146 секунды.
При увеличении размера корпуса со 100 до 750 документов словарь вырос с
4894 до 15967 терминов, а время построения индекса увеличилось с
0.034 до 0.136 секунды.

При увеличении min_len с 1 до 5 размер словаря уменьшился с 16130 до
14107 терминов, поскольку слова короче заданной длины исключались из индекса.

Для запроса 'summer wedding' оператор
AND нашёл 2 документа, а оператор OR — 38 документов.
Оператор AND подходит для поиска документов, содержащих все термины
запроса, тогда как OR позволяет найти документы, содержащие хотя бы
один из них. В основном поиске использовался оператор OR, поскольку он
обеспечивает более широкую выдачу.

По результатам сравнения поиск по инвертированному индексу занял около 8.2
микросекунды, а полный перебор документов — около 160329.6 микросекунды.
Следовательно, индекс обеспечил ускорение примерно в 19581 раз.
Главным преимуществом индекса стала высокая скорость поиска, а его цена
заключается в необходимости предварительного построения и хранения
дополнительной структуры данных, размер которой в данном случае составил
около 1001 КиБ.
"""

print(CONCLUSION)


В ходе лабораторной работы был построен инвертированный индекс для корпуса
из 750 документов. При значении min_len = 3 размер словаря составил 15967
терминов, а время построения индекса — около 0.146 секунды. 
При увеличении размера корпуса со 100 до 750 документов словарь вырос с
4894 до 15967 терминов, а время построения индекса увеличилось с 
0.034 до 0.136 секунды. 

При увеличении min_len с 1 до 5 размер словаря уменьшился с 16130 до 
14107 терминов, поскольку слова короче заданной длины исключались из индекса. 

Для запроса 'summer wedding' оператор 
AND нашёл 2 документа, а оператор OR — 38 документов. 
Оператор AND подходит для поиска документов, содержащих все термины 
запроса, тогда как OR позволяет найти документы, содержащие хотя бы 
один из них. В основном поиске использовался оператор OR, поскольку он 
обеспечивает более широкую выдачу. 

По результатам сравнения поиск по инвертированному индексу занял около 8.2 
микросекунды, а полный перебор документов — около 160329.6

In [24]:
readme = f"""# Лабораторная работа № 1
## Индексирование и булев информационный поиск

**Студент:** {STUDENT_NAME}
**Группа:** {STUDENT_GROUP}
**Вариант {VARIANT_NO}:** random_state = {RANDOM_STATE}, N = {N}, min_len = {MIN_LEN},
основной оператор — {OPERATOR}

## Метод

Корпус ({CORPUS_SOURCE}) приводится к нижнему регистру и токенизируется регулярным
выражением по латинским словам; токены короче {MIN_LEN} символов отбрасываются. Индексируется
объединённый текст `title + description + genre`. Инвертированный индекс — словарь
`термин -> множество doc_id`; документ входит в posting list термина ровно один раз.
Булев поиск: AND — пересечение posting lists, OR — объединение.

## Результаты

| Показатель | Значение |
|---|---:|
| Источник корпуса | {CORPUS_SOURCE} |
| Всего документов в корпусе | {len(docs_all)} |
| Число документов N | {len(docs)} |
| random_state | {RANDOM_STATE} |
| min_len | {MIN_LEN} |
| Размер словаря | {len(index)} |
| Время построения индекса, с | {BUILD_TIME:.6f} |
| Запрос | `{QUERY}` |
| Число результатов AND | {len(ids_and)} |
| Время AND, с | {t_and:.8f} |
| Число результатов OR | {len(ids_or)} |
| Время OR, с | {t_or:.8f} |
| Время полного перебора, с | {t_linear:.8f} |
| Ускорение за счёт индекса | {t_linear / t_index:.0f}x |

### Словарь в зависимости от min_len

| min_len | Размер словаря |
|---:|---:|
""" + "\n".join(f"| {k} | {v[0]} |" for k, v in sorted(by_min_len.items())) + f"""

### Словарь в зависимости от размера корпуса

| N | Размер словаря |
|---:|---:|
""" + "\n".join(f"| {k} | {v[0]} |" for k, v in sorted(by_size.items())) + f"""

## Вывод
{CONCLUSION}

## Воспроизведение

Ноутбук `lab_01.ipynb` открывается в Google Colab и выполняется сверху вниз без установки
зависимостей: используется только стандартная библиотека Python. Корпус собирается автоматически.
Для точного воспроизведения задайте `VARIANT_NO = {VARIANT_NO}`.
"""

(WORK / "README.md").write_text(readme, encoding="utf-8")
(WORK / "requirements.txt").write_text(
    "# Решение использует только стандартную библиотеку Python 3.12.\n"
    "# Дополнительные пакеты не требуются.\n", encoding="utf-8")

import zipfile
zip_path = WORK / "lab_01.zip"
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as z:
    for name in ["README.md", "requirements.txt", "index.json", "lab_01_results.json"]:
        p = WORK / name
        if p.exists():
            z.write(p, arcname=f"lab_01/{name}")

print("Готово. Файлы для репозитория:")
for name in ["README.md", "requirements.txt", "index.json", "lab_01_results.json", "lab_01.zip"]:
    p = WORK / name
    if p.exists():
        print(f"  {name:<24} {p.stat().st_size / 1024:8.1f} КиБ")

from google.colab import files
files.download(str(zip_path))

Готово. Файлы для репозитория:
  README.md                     4.7 КиБ
  requirements.txt              0.2 КиБ
  index.json                 1001.0 КиБ
  lab_01_results.json           6.3 КиБ
  lab_01.zip                  222.8 КиБ


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>